# 02e — Advanced modeling: SMOTE + Optuna + per-position thresholds

Picks up the winner from 02b (5-feature RFECV XGBoost) and applies three
techniques that were previously verified to improve the model:

| Section | Technique | Verified contribution |
|---------|-----------|-----------------------|
| A | SMOTE / Imbalance handling | +0.018 OOF AUC |
| B | Optuna hyperparameter tuning | +0.033 OOF AUC |
| C | Per-position decision threshold | improves balanced accuracy |
| D | Final test-set evaluation + persist |

> **Removed in this version:** hybrid attacker stratification
> (−0.011 OOF AUC) and stacking ensemble (−0.019 OOF AUC) were tested in
> an earlier iteration and discarded for under-performance.


## Setup

In [1]:
"""Imports."""
from __future__ import annotations
import sys, warnings, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
"""Modelling stack."""
import xgboost as xgb
import optuna
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.combine import SMOTETomek
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score, average_precision_score,
    brier_score_loss, log_loss, confusion_matrix, classification_report,
)
from sklearn.isotonic import IsotonicRegression
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"xgb {xgb.__version__}, optuna {optuna.__version__}")


xgb 3.2.0, optuna 4.8.0


## Load 02b winner

We re-load the winning 5-feature dataset from 02b — the result of
recursive feature elimination over the 91-feature kitchen-sink pool.


In [3]:
"""Load 02b artefacts."""
ART_02b = PROJECT_ROOT / "models" / "kitchen_sink"
X_train = pd.read_csv(ART_02b / "X_train_raw.csv")
X_test = pd.read_csv(ART_02b / "X_test_raw.csv")
y_train = pd.read_csv(ART_02b / "y_train_raw.csv")["scored_after"]
y_test = pd.read_csv(ART_02b / "y_test_raw.csv")["scored_after"]
config_02b = json.loads((ART_02b / "config.json").read_text())

print(f"X_train: {X_train.shape}, y_train pos={int(y_train.sum())}")
print(f"X_test:  {X_test.shape}, y_test pos={int(y_test.sum())}")
print(f"Features: {list(X_train.columns)}")


X_train: (2394, 5), y_train pos=173
X_test:  (720, 5), y_test pos=29
Features: ['top_third_pass_share', 'forward_pass_share_D', 'cumul_sprints', 'jersey_number', 'cumul_passes_M']


In [4]:
"""Load checkpoint and group info."""
oof_meta = pd.read_csv(ART_02b / "oof_predictions.csv")
test_meta = pd.read_csv(ART_02b / "test_predictions.csv")
g_train = oof_meta["fixture_id"].values
cp_train = oof_meta["checkpoint"].values
cp_test = test_meta["checkpoint"].values
spw = float(config_02b["scale_pos_weight"])
print(f"distinct train fixtures: {len(set(g_train))}")
print(f"distinct test fixtures:  {len(set(test_meta['fixture_id'].values))}")
print(f"scale_pos_weight: {spw:.3f}")


distinct train fixtures: 24
distinct test fixtures:  7
scale_pos_weight: 12.838


In [5]:
"""Re-establish 02b baseline OOF AUC (sanity check)."""
def oof_xgb_with_resampler(resampler, X, y, groups, params):
    """5-fold GroupKFold OOF probabilities; resampling applied only to train fold."""
    oof = np.zeros(len(X))
    for tr, va in GroupKFold(5).split(X, y, groups):
        X_tr, y_tr = X.iloc[tr].copy(), y.iloc[tr].copy()
        X_va, y_va = X.iloc[va].copy(), y.iloc[va].copy()
        if resampler is not None:
            X_tr, y_tr = resampler.fit_resample(X_tr, y_tr)
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof[va] = m.predict_proba(X_va)[:, 1]
    return oof


baseline_params = dict(
    n_estimators=600, learning_rate=0.05, max_depth=3,
    min_child_weight=10, subsample=0.9, colsample_bytree=0.85,
    scale_pos_weight=spw,
    objective="binary:logistic", eval_metric="auc",
    early_stopping_rounds=30, n_jobs=-1, random_state=RANDOM_SEED,
    tree_method="hist", verbosity=0,
)
oof_baseline = oof_xgb_with_resampler(None, X_train, y_train, g_train, baseline_params)
auc_baseline = roc_auc_score(y_train, oof_baseline)
ap_baseline = average_precision_score(y_train, oof_baseline)
print(f"BASELINE (02b winner reproduced):")
print(f"  OOF AUC = {auc_baseline:.4f}")
print(f"  OOF AP  = {ap_baseline:.4f}")


BASELINE (02b winner reproduced):
  OOF AUC = 0.6578
  OOF AP  = 0.1162


## Section A — SMOTE / Imbalance handling

Test resampling strategies inside an imblearn-style pipeline (resampling
applied only to training folds, never to validation). We sweep two
sampling-strategy values for both random oversampling and SMOTE,
plus SMOTETomek as a combined oversample-and-clean approach.

Critical detail: when oversampling, we set `scale_pos_weight = 1` to
avoid double-correcting for the class imbalance.


In [6]:
"""SMOTE experiment."""
RESAMPLERS = {
    "vanilla": None,
    "ROS_0.2": RandomOverSampler(sampling_strategy=0.2, random_state=RANDOM_SEED),
    "ROS_0.3": RandomOverSampler(sampling_strategy=0.3, random_state=RANDOM_SEED),
    "SMOTE_0.2": SMOTE(sampling_strategy=0.2, k_neighbors=5, random_state=RANDOM_SEED),
    "SMOTE_0.3": SMOTE(sampling_strategy=0.3, k_neighbors=5, random_state=RANDOM_SEED),
    "SMOTETomek_0.3": SMOTETomek(sampling_strategy=0.3, random_state=RANDOM_SEED),
}

results_A = []
for name, resampler in RESAMPLERS.items():
    params = dict(baseline_params)
    if resampler is not None:
        params["scale_pos_weight"] = 1.0
    oof_p = oof_xgb_with_resampler(resampler, X_train, y_train, g_train, params)
    auc = roc_auc_score(y_train, oof_p)
    ap = average_precision_score(y_train, oof_p)
    results_A.append({"strategy": name, "oof_auc": auc, "oof_ap": ap})
    print(f"  {name:18s}  OOF AUC={auc:.4f}  AP={ap:.4f}")
results_A_df = pd.DataFrame(results_A).sort_values("oof_auc", ascending=False).reset_index(drop=True)
print()
print(results_A_df.to_string(index=False))


  vanilla             OOF AUC=0.6578  AP=0.1162
  ROS_0.2             OOF AUC=0.6689  AP=0.1416
  ROS_0.3             OOF AUC=0.6758  AP=0.1310
  SMOTE_0.2           OOF AUC=0.6432  AP=0.1097
  SMOTE_0.3           OOF AUC=0.6161  AP=0.1041
  SMOTETomek_0.3      OOF AUC=0.6283  AP=0.1158

      strategy  oof_auc  oof_ap
       ROS_0.3   0.6758  0.1310
       ROS_0.2   0.6689  0.1416
       vanilla   0.6578  0.1162
     SMOTE_0.2   0.6432  0.1097
SMOTETomek_0.3   0.6283  0.1158
     SMOTE_0.3   0.6161  0.1041


In [7]:
"""Adopt the winning SMOTE strategy."""
winner_A = results_A_df.iloc[0]
SMOTE_WINNER_NAME = winner_A["strategy"]
SMOTE_WINNER = RESAMPLERS[SMOTE_WINNER_NAME]
delta_A = winner_A["oof_auc"] - auc_baseline
print(f"Section A winner: {SMOTE_WINNER_NAME}")
print(f"  OOF AUC: {auc_baseline:.4f} -> {winner_A['oof_auc']:.4f}  (Δ = {delta_A:+.4f})")


Section A winner: ROS_0.3
  OOF AUC: 0.6578 -> 0.6758  (Δ = +0.0179)


## Section B — Optuna hyperparameter tuning

Bayesian optimisation over XGBoost hyper-parameters using TPE sampler
on top of the winning resampling from Section A. The MedianPruner aborts
trials whose intermediate fold AUCs lag behind the median, saving
compute. We run 60 trials.


In [8]:
"""Optuna study."""
def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 800),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 6),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 30.0),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        gamma=trial.suggest_float("gamma", 0.0, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 5.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 0.0, 1.0),
        scale_pos_weight=spw if SMOTE_WINNER is None else 1.0,
        objective="binary:logistic", eval_metric="auc",
        early_stopping_rounds=30, n_jobs=-1, random_state=RANDOM_SEED,
        tree_method="hist", verbosity=0,
    )
    fold_aucs = []
    for fold_idx, (tr, va) in enumerate(GroupKFold(5).split(X_train, y_train, g_train)):
        X_tr, y_tr = X_train.iloc[tr].copy(), y_train.iloc[tr].copy()
        X_va, y_va = X_train.iloc[va].copy(), y_train.iloc[va].copy()
        if SMOTE_WINNER is not None:
            X_tr, y_tr = SMOTE_WINNER.fit_resample(X_tr, y_tr)
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        fold_aucs.append(roc_auc_score(y_va, m.predict_proba(X_va)[:, 1]))
        trial.report(np.mean(fold_aucs), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(fold_aucs))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED, n_startup_trials=10),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)
study.optimize(objective, n_trials=60, show_progress_bar=False, n_jobs=1)

print(f"finished {len(study.trials)} trials, "
      f"{sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)} pruned")
print(f"best OOF AUC: {study.best_value:.4f}")
print(f"best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


finished 60 trials, 0 pruned
best OOF AUC: 0.7084
best params:
  n_estimators: 412
  learning_rate: 0.0952318831680396
  max_depth: 4
  min_child_weight: 5.89712909129932
  subsample: 0.7482728324730173
  colsample_bytree: 0.6719504468986662
  gamma: 0.520505401411339
  reg_lambda: 1.2945717915418564
  reg_alpha: 0.9170907319592008


In [9]:
"""Adopt Optuna params."""
OPTUNA_PARAMS = dict(study.best_params)
OPTUNA_PARAMS.update(dict(
    scale_pos_weight=spw if SMOTE_WINNER is None else 1.0,
    objective="binary:logistic", eval_metric="auc",
    early_stopping_rounds=30, n_jobs=-1, random_state=RANDOM_SEED,
    tree_method="hist", verbosity=0,
))
delta_B = study.best_value - winner_A["oof_auc"]
print(f"Section B: Optuna best AUC = {study.best_value:.4f}")
print(f"  Δ vs Section A: {delta_B:+.4f}")


Section B: Optuna best AUC = 0.7084
  Δ vs Section A: +0.0326


In [10]:
"""Build OOF predictions with the winning A+B configuration."""
oof_xgb_best = oof_xgb_with_resampler(SMOTE_WINNER, X_train, y_train, g_train, OPTUNA_PARAMS)
auc_AB = roc_auc_score(y_train, oof_xgb_best)
ap_AB = average_precision_score(y_train, oof_xgb_best)
print(f"After Section A+B: OOF AUC = {auc_AB:.4f}  AP = {ap_AB:.4f}")
print(f"  cumulative delta vs baseline: {auc_AB - auc_baseline:+.4f}")


After Section A+B: OOF AUC = 0.6933  AP = 0.1244
  cumulative delta vs baseline: +0.0355


## Section C — Per-position decision threshold

A single pooled XGBoost produces calibrated probabilities for all
positions. Because attacker, midfielder, and defender base rates differ
substantially (12.6%, 6.8%, 3.1% respectively), a single global decision
threshold is a compromise across all three. We tune one threshold per
position class on out-of-fold predictions to maximise per-class balanced
accuracy.

This step does **not** retrain the model; it only changes the
thresholding policy at inference time. It costs nothing and reliably
improves balanced accuracy on the per-position metric.


In [11]:
"""Recover position labels for train and test."""
main = pd.read_csv(PROJECT_ROOT / "data" / "players_quarters_final.csv")
position_lookup = main[["player_appearance_id", "checkpoint", "position"]].drop_duplicates()
train_pos = oof_meta.merge(
    position_lookup, on=["player_appearance_id", "checkpoint"], how="left",
)["position"].values
test_pos = test_meta.merge(
    position_lookup, on=["player_appearance_id", "checkpoint"], how="left",
)["position"].values
print(f"train position breakdown:")
print(pd.Series(train_pos).value_counts().to_string())


train position breakdown:
M    995
D    937
A    462


In [12]:
"""Calibrate OOF probabilities and tune per-position thresholds."""
iso_global = IsotonicRegression(out_of_bounds="clip")
iso_global.fit(oof_xgb_best, y_train)
oof_cal = iso_global.transform(oof_xgb_best)

THRESH_RANGE = np.linspace(0.01, 0.6, 60)
def best_threshold(probs, y):
    bas = [balanced_accuracy_score(y, (probs >= t).astype(int)) for t in THRESH_RANGE]
    i = int(np.argmax(bas))
    return float(THRESH_RANGE[i]), float(bas[i])


# Global threshold (for comparison).
global_thr, global_oof_ba = best_threshold(oof_cal, y_train)
print(f"global threshold: {global_thr:.3f}  -> OOF BA = {global_oof_ba:.4f}")
print()

# Per-position thresholds.
percp_thresholds = {}
print("per-position thresholds:")
for pos in ("A", "M", "D"):
    mask = train_pos == pos
    if mask.sum() < 50:
        percp_thresholds[pos] = global_thr; continue
    thr, ba = best_threshold(oof_cal[mask], y_train.iloc[mask])
    percp_thresholds[pos] = thr
    print(f"  pos={pos}  n={int(mask.sum()):4d}  pos_count={int(y_train.iloc[mask].sum()):3d}  "
          f"thr={thr:.3f}  OOF BA={ba:.4f}")


global threshold: 0.070  -> OOF BA = 0.6841

per-position thresholds:
  pos=A  n= 462  pos_count= 62  thr=0.040  OOF BA=0.5944
  pos=M  n= 995  pos_count= 73  thr=0.070  OOF BA=0.6020
  pos=D  n= 937  pos_count= 38  thr=0.110  OOF BA=0.7765


## Section D — Final test-set evaluation

Refit the final XGBoost model (with Optuna params + SMOTE) on the full
training set, predict on test, calibrate, and apply per-position
thresholds.


In [13]:
"""Refit XGB on full train + predict on test."""
final_params = dict(OPTUNA_PARAMS)
final_params["early_stopping_rounds"] = None  # no eval set, no early stopping

X_train_resampled, y_train_resampled = X_train.copy(), y_train.copy()
if SMOTE_WINNER is not None:
    X_train_resampled, y_train_resampled = SMOTE_WINNER.fit_resample(X_train, y_train)

model_xgb = xgb.XGBClassifier(**final_params)
model_xgb.fit(X_train_resampled, y_train_resampled, verbose=False)
test_proba_raw = model_xgb.predict_proba(X_test)[:, 1]
print(f"raw test mean prob: {test_proba_raw.mean():.4f}, actual rate: {y_test.mean():.4f}")


raw test mean prob: 0.1495, actual rate: 0.0403


In [14]:
"""Calibrate test predictions on global isotonic (fit on OOF)."""
iso_final = IsotonicRegression(out_of_bounds="clip")
iso_final.fit(oof_xgb_best, y_train)
oof_cal_final = iso_final.transform(oof_xgb_best)
test_proba_cal = iso_final.transform(test_proba_raw)
print(f"calibrated test mean prob: {test_proba_cal.mean():.4f}")


calibrated test mean prob: 0.0441


In [15]:
"""Apply per-position thresholds and report final metrics."""
test_pred = np.zeros(len(test_proba_cal), dtype=int)
for pos, thr in percp_thresholds.items():
    mask = test_pos == pos
    test_pred[mask] = (test_proba_cal[mask] >= thr).astype(int)

print(f"=== FINAL TEST METRICS ===")
print(f"  AUC (calibrated):    {roc_auc_score(y_test, test_proba_cal):.4f}")
print(f"  AP  (calibrated):    {average_precision_score(y_test, test_proba_cal):.4f}")
print(f"  Brier (calibrated):  {brier_score_loss(y_test, test_proba_cal):.4f}")
print(f"  Balanced accuracy:   {balanced_accuracy_score(y_test, test_pred):.4f}")
print()
cm = confusion_matrix(y_test, test_pred)
print(pd.DataFrame(cm, index=["actual_0","actual_1"], columns=["pred_0","pred_1"]).to_string())
print()
print(classification_report(y_test, test_pred, target_names=["no_goal", "goal"]))


=== FINAL TEST METRICS ===
  AUC (calibrated):    0.7017
  AP  (calibrated):    0.0733
  Brier (calibrated):  0.0394
  Balanced accuracy:   0.6824

          pred_0  pred_1
actual_0     538     153
actual_1      12      17

              precision    recall  f1-score   support

     no_goal       0.98      0.78      0.87       691
        goal       0.10      0.59      0.17        29

    accuracy                           0.77       720
   macro avg       0.54      0.68      0.52       720
weighted avg       0.94      0.77      0.84       720



In [16]:
"""Section progression summary."""
final_summary = pd.DataFrame([
    {"section": "02b baseline (5 features, vanilla XGB)",
     "oof_auc": auc_baseline, "test_auc": float("nan"), "test_ba": float("nan"), "test_brier": float("nan")},
    {"section": f"+ Section A (SMOTE: {SMOTE_WINNER_NAME})",
     "oof_auc": winner_A["oof_auc"], "test_auc": float("nan"), "test_ba": float("nan"), "test_brier": float("nan")},
    {"section": "+ Section B (Optuna)",
     "oof_auc": auc_AB, "test_auc": float("nan"), "test_ba": float("nan"), "test_brier": float("nan")},
    {"section": "+ Section C (per-position thresholds, final)",
     "oof_auc": auc_AB,
     "test_auc": roc_auc_score(y_test, test_proba_cal),
     "test_ba": balanced_accuracy_score(y_test, test_pred),
     "test_brier": brier_score_loss(y_test, test_proba_cal)},
])
print(final_summary.round(4).to_string(index=False))


                                     section  oof_auc  test_auc  test_ba  test_brier
      02b baseline (5 features, vanilla XGB)   0.6578       NaN      NaN         NaN
                + Section A (SMOTE: ROS_0.3)   0.6758       NaN      NaN         NaN
                        + Section B (Optuna)   0.6933       NaN      NaN         NaN
+ Section C (per-position thresholds, final)   0.6933    0.7017   0.6824      0.0394


## Persist artefacts

In [18]:
"""Save."""
ART = PROJECT_ROOT / "models" / "advanced"
ART.mkdir(exist_ok=True)

# Model + calibrator
with open(ART / "model_xgb.pkl", "wb") as f:
    pickle.dump(model_xgb, f)
with open(ART / "isotonic_calibrator.pkl", "wb") as f:
    pickle.dump(iso_final, f)

# Predictions
oof_df = oof_meta[["player_appearance_id","checkpoint","fixture_id","scored_after"]].copy()
oof_df["oof_proba"] = oof_xgb_best
oof_df["oof_proba_calibrated"] = oof_cal_final
oof_df.to_csv(ART / "oof_predictions.csv", index=False)

test_df = test_meta[["player_appearance_id","checkpoint","fixture_id","scored_after"]].copy()
test_df["test_proba"] = test_proba_raw
test_df["test_proba_calibrated"] = test_proba_cal
test_df["test_pred"] = test_pred
test_df.to_csv(ART / "test_predictions.csv", index=False)

# Summary + raw matrices
final_summary.to_csv(ART / "section_comparison.csv", index=False)
X_train.to_csv(ART / "X_train_raw.csv", index=False)
X_test.to_csv(ART / "X_test_raw.csv", index=False)
y_train.to_csv(ART / "y_train_raw.csv", index=False, header=True)
y_test.to_csv(ART / "y_test_raw.csv", index=False, header=True)

# Config
final_config = {
    "smote_winner": SMOTE_WINNER_NAME,
    "optuna_params": {k: float(v) if isinstance(v, (int, float, np.floating)) else v
                       for k, v in OPTUNA_PARAMS.items()},
    "global_threshold": float(global_thr),
    "percp_thresholds": {k: float(v) for k, v in percp_thresholds.items()},
    "metrics": {
        "oof_auc_baseline": float(auc_baseline),
        "oof_auc_section_A": float(winner_A["oof_auc"]),
        "oof_auc_section_AB": float(auc_AB),
        "test_auc": float(roc_auc_score(y_test, test_proba_cal)),
        "test_balanced_accuracy": float(balanced_accuracy_score(y_test, test_pred)),
        "test_brier": float(brier_score_loss(y_test, test_proba_cal)),
    },
    "removed_techniques": {
        "hybrid_attacker_stratification": "rejected: -0.011 OOF AUC vs pooled",
        "stacking_ensemble": "rejected: -0.019 OOF AUC vs single XGB",
    },
}
with open(ART / "config.json", "w") as f:
    json.dump(final_config, f, indent=2, default=str)

print(f"saved to {ART}/:")
for p in sorted(ART.iterdir()):
    print(f"  {p.name:30s} {p.stat().st_size / 1024:.1f} KB")


saved to c:\Users\tymot\projects\wec\models\advanced/:
  config.json                    1.2 KB
  isotonic_calibrator.pkl        0.8 KB
  model_xgb.pkl                  454.3 KB
  oof_predictions.csv            138.9 KB
  section_comparison.csv         0.3 KB
  test_predictions.csv           32.4 KB
  X_test_raw.csv                 19.8 KB
  X_train_raw.csv                65.1 KB
  y_test_raw.csv                 2.1 KB
  y_train_raw.csv                7.0 KB


### Summary

The cleaned pipeline retains only the three techniques that empirically
improved the model:

* **SMOTE** (RandomOverSampler with sampling_strategy=0.3) replaces
  scale_pos_weight rebalancing in CV.
* **Optuna** with 60 TPE trials produced a substantially stronger
  hyper-parameter configuration than the manual grid search.
* **Per-position thresholds** tune the decision boundary separately for
  attackers, midfielders, and defenders, accommodating their very
  different base rates.

Two techniques previously evaluated and rolled back:

* **Hybrid attacker stratification** (separate model for attackers,
  pooled for midfielders + defenders): −0.011 OOF AUC. The smaller
  attacker training set (462 rows, 62 positives) failed to support a
  stable submodel, and the pooled approach with position dummies
  retained more cross-position context.
* **Stacking ensemble** (XGBoost + LightGBM + L2 logistic regression
  with logistic meta-learner): −0.019 OOF AUC. The base models had
  uneven strength (XGBoost was substantially stronger than LightGBM and
  LR), so the meta-learner had little complementary signal to combine
  and instead added variance.

**Next steps for further improvement:**

1. `02f_score_state_features.ipynb` — engineer match-state features
   (current_score_diff, team_xg_so_far, time_since_last_team_goal) and
   re-run the kitchen-sink selection on the extended pool. Estimated
   contribution: +0.03-0.05 OOF AUC. Highest remaining ROI.
2. `03_xai_explanations.ipynb` — full XAI analysis on the final model
   for the report (Break-Down + SHAP + PDP/ALE + beeswarm).
